# Deepfake detection on OpenForensics (Colab)

Fine-tunes a pretrained ResNet-18 to classify face crops as **real** or **fake**.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

Steps: download data -> crop faces -> train -> evaluate.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable a GPU runtime")

## 1. Download the data

Downloads from [Zenodo](https://zenodo.org/records/5528418) (OpenForensics, CC BY 4.0) to Colab's local disk. Downloads resume if interrupted and are checked against Zenodo's MD5. Zips are deleted after extraction. The disk is erased when the runtime disconnects, so rerun this each session.

Edit `FILES_TO_GET` to change what is downloaded (Train parts are ~4 GB each).

In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import urllib.request
import zipfile
from pathlib import Path

from tqdm.auto import tqdm

RECORD_ID = "5528418"
DATA_DIR = Path("/content/data/OpenForensics")
ZIP_DIR = Path("/content/zips")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

FILES_TO_GET = [
    "readme.txt",
    "Val.zip",
    "Val_poly.json",
    # "Train_part_1.zip",
    # "Train_poly.json",
]

with urllib.request.urlopen(f"https://zenodo.org/api/records/{RECORD_ID}") as r:
    files = {f["key"]: f for f in json.load(r)["files"]}

total = sum(files[n]["size"] for n in FILES_TO_GET)
print(f"Selected {len(FILES_TO_GET)} files, {total / 1e9:.1f} GB "
      f"({shutil.disk_usage('/content').free / 1e9:.0f} GB free on Colab disk)")


def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


if shutil.which("aria2c") is None:   # multi-connection downloader, much faster than one stream
    subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], check=True)


def download(info, dest):
    size, url = info["size"], info["links"]["self"]
    have = dest.stat().st_size if dest.exists() else 0
    if have == size and not Path(f"{dest}.aria2").exists():
        return
    # aria2c resumes with -c. We read its progress readout, e.g. "[#a1b2c3 1234/3000000(0%) CN:8 ...]",
    # and drive a normal tqdm bar from the byte counts.
    proc = subprocess.Popen(
        ["aria2c", "-c", "-x", "8", "-s", "8", "-k", "4M", "--human-readable=false",
         "--console-log-level=error", "--summary-interval=0", "-d", str(dest.parent),
         "-o", dest.name, url],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    progress = re.compile(rb"\[#\w+ (\d+)/(\d+)\(\d+%\)")
    tail = b""
    with tqdm(total=size, unit="B", unit_scale=True, desc=dest.name) as bar:
        while chunk := os.read(proc.stdout.fileno(), 4096):
            tail = (tail + chunk)[-4096:]
            found = progress.findall(tail)
            if found:
                bar.update(min(int(found[-1][0]), size) - bar.n)
        proc.wait()
        if proc.returncode == 0:
            bar.update(size - bar.n)
    if proc.returncode != 0:
        raise RuntimeError(f"aria2c failed for {dest.name}:\n{tail.decode(errors='replace')}")


def extract(zip_path, out_dir):
    with zipfile.ZipFile(zip_path) as z:
        n = len(z.namelist())
    # command-line unzip is faster than Python's zipfile; each extracted file prints one line
    proc = subprocess.Popen(["unzip", "-o", str(zip_path), "-d", str(out_dir)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in tqdm(proc.stdout, total=n, desc=f"Extracting {zip_path.name}"):
        pass
    if proc.wait() != 0:
        raise RuntimeError(f"unzip failed for {zip_path.name}")


for name in FILES_TO_GET:
    info = files[name]
    dest = (ZIP_DIR if name.endswith(".zip") else DATA_DIR) / name
    marker = DATA_DIR / f".{name}.done"
    if marker.exists():
        print(f"{name}: already downloaded and extracted")
        continue

    for _ in range(3):
        download(info, dest)
        if md5sum(dest) == info["checksum"].split(":", 1)[1]:
            print(f"{name}: checksum OK")
            break
        print(f"{name}: checksum mismatch, re-downloading")
        dest.unlink()
        Path(f"{dest}.aria2").unlink(missing_ok=True)
    else:
        raise RuntimeError(f"{name} failed checksum 3 times")

    if name.endswith(".zip"):
        extract(dest, DATA_DIR)
        dest.unlink()
    marker.touch()

print("Done:", [p.name for p in sorted(DATA_DIR.iterdir()) if not p.name.startswith(".")])

## 2. Build the face dataset

Each annotation is one face (COCO-style box + real/fake label). We crop every face once, resize it, and keep everything in memory so training is fast.

The train/val/test split is done **by image**, so faces from the same photo never end up in different splits.

In [ ]:
import json, random
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

SIZE = 128     # crop size fed to the network
PAD = 0.2       # extra context around the face box

with open(DATA_DIR / "Val_poly.json") as f:
    coco = json.load(f)

print("Categories:", coco["categories"])
# Label 1 = fake, 0 = real, decided from the category names
fake_ids = {c["id"] for c in coco["categories"] if "fake" in c["name"].lower()}
assert fake_ids, "Could not find a 'fake' category, check the printout above"

images = {im["id"]: im for im in coco["images"]}
paths = {p.name: p for p in DATA_DIR.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
print(f"{len(images)} images in annotations, {len(paths)} image files on disk")

by_image = {}
for a in coco["annotations"]:
    by_image.setdefault(a["image_id"], []).append(a)

In [ ]:
random.seed(0)
ids = sorted(by_image)
random.shuffle(ids)
n = len(ids)
split_of = {}
for i, img_id in enumerate(ids):
    split_of[img_id] = "train" if i < 0.7 * n else "val" if i < 0.85 * n else "test"

data = {s: ([], []) for s in ("train", "val", "test")}   # (crops, labels)

for img_id in tqdm(ids, desc="Cropping faces"):
    im_info = images[img_id]
    path = paths.get(Path(im_info["file_name"]).name)
    if path is None:
        continue
    img = Image.open(path).convert("RGB")
    W, H = img.size
    for a in by_image[img_id]:
        x, y, w, h = a["bbox"]
        if w < 8 or h < 8:
            continue
        x0, y0 = max(0, x - PAD * w), max(0, y - PAD * h)
        x1, y1 = min(W, x + (1 + PAD) * w), min(H, y + (1 + PAD) * h)
        crop = img.crop((int(x0), int(y0), int(x1), int(y1))).resize((SIZE, SIZE), Image.BILINEAR)
        crops, labels = data[split_of[img_id]]
        crops.append(np.asarray(crop))
        labels.append(int(a["category_id"] in fake_ids))

tensors = {}
for s, (crops, labels) in data.items():
    X = torch.from_numpy(np.stack(crops)).permute(0, 3, 1, 2).contiguous()   # uint8, N x 3 x H x W
    y = torch.tensor(labels)
    tensors[s] = (X, y)
    print(f"{s:5s}: {len(y):6d} faces, {int(y.sum()):6d} fake ({y.float().mean():.0%})")

## 3. Train

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS, BATCH, LR = 5, 64, 1e-3

mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def prep(xb, train):
    xb = xb.to(device, non_blocking=True).float() / 255
    if train:   # random horizontal flip
        flip = torch.rand(xb.size(0), device=device) < 0.5
        xb = torch.where(flip.view(-1, 1, 1, 1), xb.flip(3), xb)
    return (xb - mean) / std

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model.to(device)

# Weight classes so a real/fake imbalance doesn't dominate the loss
y_train = tensors["train"][1]
counts = torch.bincount(y_train, minlength=2).float()
class_w = (counts.sum() / (2 * counts)).to(device)

opt = torch.optim.AdamW(model.parameters(), lr=LR)
steps = EPOCHS * ((len(y_train) + BATCH - 1) // BATCH)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps)
scaler = torch.amp.GradScaler(enabled=(device == "cuda"))

@torch.no_grad()
def predict(split):
    model.eval()
    X, y = tensors[split]
    probs = []
    for i in range(0, len(y), 256):
        with torch.autocast(device, enabled=(device == "cuda")):
            logits = model(prep(X[i:i + 256], False))
        probs.append(logits.float().softmax(1)[:, 1].cpu())
    return torch.cat(probs), y

history, best_acc = [], 0
for epoch in range(EPOCHS):
    model.train()
    X, y = tensors["train"]
    perm = torch.randperm(len(y))
    total = 0
    for i in tqdm(range(0, len(y), BATCH), desc=f"Epoch {epoch + 1}/{EPOCHS}", leave=False):
        idx = perm[i:i + BATCH]
        xb, yb = prep(X[idx], True), y[idx].to(device)
        with torch.autocast(device, enabled=(device == "cuda")):
            loss = F.cross_entropy(model(xb), yb, weight=class_w)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        total += loss.item() * len(idx)

    p, yv = predict("val")
    acc = ((p > 0.5).long() == yv).float().mean().item()
    history.append((total / len(y), acc))
    print(f"epoch {epoch + 1}: train loss {total / len(y):.4f} | val acc {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "best.pt")

model.load_state_dict(torch.load("best.pt"))

## 4. Evaluate on the held-out test split

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# Everything saved here: copy the folder to Drive to keep it, or point RESULTS_DIR at
# "/content/drive/MyDrive/deepfake_results" after mounting Drive.
RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

p, y = predict("test")
pred = (p > 0.5).long()
print(f"Test accuracy: {accuracy_score(y, pred):.4f}")
print(f"Test ROC AUC:  {roc_auc_score(y, p):.4f}")
report = classification_report(y, pred, target_names=["real", "fake"])
print(report)
(RESULTS_DIR / "metrics.txt").write_text(
    f"Test accuracy: {accuracy_score(y, pred):.4f}\n"
    f"Test ROC AUC:  {roc_auc_score(y, p):.4f}\n\n{report}")

cm = confusion_matrix(y, pred)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(cm, cmap="Blues")
ax[0].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["real", "fake"], yticklabels=["real", "fake"],
          xlabel="predicted", ylabel="actual", title="Confusion matrix")
for (i, j), v in np.ndenumerate(cm):
    ax[0].text(j, i, v, ha="center", va="center")
ax[1].plot([h[0] for h in history], label="train loss")
ax[1].plot([h[1] for h in history], label="val acc")
ax[1].set(xlabel="epoch", title="Training"); ax[1].legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_and_training.png", dpi=150, bbox_inches="tight")
plt.show()

### Demo: example predictions on the test split

Random face crops from the test split. The title shows the true label, the model's guess and its fake-probability. Green = correct, red = wrong. Rerun the cell for a new random sample.

In [ ]:
import random as _random

LABELS = ["real", "fake"]
N = 16
Xt, yt = tensors["test"]
probs_t, _ = predict("test")

demo_idx = _random.sample(range(len(yt)), N)
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for ax, i in zip(axes.flat, demo_idx):
    p_fake = probs_t[i].item()
    guess = int(p_fake > 0.5)
    truth = int(yt[i])
    ax.imshow(Xt[i].permute(1, 2, 0).numpy())
    ax.set_title(f"true: {LABELS[truth]}\npred: {LABELS[guess]} ({p_fake:.0%} fake)",
                 color="green" if guess == truth else "red", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "demo_predictions.png", dpi=150, bbox_inches="tight")
plt.show()